In [0]:
# 1. Which student scored max marks in each semester considering all subjects
# 2. Percentage of each student considering all subjects
# 3. Who is the top rank holder in each semester considering all subjects
# 4. Who scored max marks in each subject in each semester

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

data = [
    ("Alice", "Math", 90, 1),
    ("Alice", "Science", 85, 1),
    ("Alice", "History", 78, 1),
    ("Bob", "Math", 80, 1),
    ("Bob", "Science", 81, 1),
    ("Bob", "History", 77, 1),
    ("Charlie", "Math", 75, 1),
    ("Charlie", "Science", 82, 1),
    ("Charlie", "History", 79, 1),
    ("Alice", "Physics", 86, 2),
    ("Alice", "Chemistry", 92, 2),
    ("Alice", "Biology", 80, 2),
    ("Bob", "Physics", 94, 2),
    ("Bob", "Chemistry", 91, 2),
    ("Bob", "Biology", 96, 2),
    ("Charlie", "Physics", 89, 2),
    ("Charlie", "Chemistry", 88, 2),
    ("Charlie", "Biology", 85, 2),
    ("Alice", "Computer Science", 95, 3),
    ("Alice", "Electronics", 91, 3),
    ("Alice", "Geography", 97, 3),
    ("Bob", "Computer Science", 88, 3),
    ("Bob", "Electronics", 66, 3),
    ("Bob", "Geography", 92, 3),
    ("Charlie", "Computer Science", 92, 3),
    ("Charlie", "Electronics", 97, 3),
    ("Charlie", "Geography", 99, 3)
]

columns = ["First Name", "Subject", "Marks", "Semester"]
df = spark.createDataFrame(data,columns)
df.show()

In [0]:
# 1. Which student scored max marks in each semester considering all subjects
window_spec = Window.partitionBy("Semester").orderBy(desc("Marks"))
max_marks = df.withColumn("Rank",rank().over(window_spec))
result = max_marks.filter(col("Rank") == 1)
result.show()

In [0]:
# 2. Percentage of each student considering all subjects

window_spec = Window.partitionBy("First Name","Semester")
df1 = df.withColumn("TotalMarks",sum("Marks").over(window_spec))
df1 = df1.withColumn("Percentage",(col("TotalMarks")/(3*100)).cast("decimal(5,2)")*100)
df2 = df1.groupBy(col("First Name"),col("Semester"))\
    .agg(max(col("TotalMarks")).alias("TotalMarks"),
         max(col("Percentage")).alias("percentage"))
display(df2)
    